В этом файле речь пойдет об очистке и предварительной обработке данных.

Первым делом мы привели все колонки к корректным типам. Затем проанализировали пропуски. Важно было не просто посчитать их количество, но и понять природу: связаны ли они с другими переменными или случайны. Для каждой проблемной колонки мы проверили, отличается ли средняя цена товаров с пропусками от цены товаров без пропусков. Например, если у товаров без указанной страны производства цена систематически ниже — значит, пропуски не случайны, и просто так их удалять нельзя. Такой анализ помог выбрать правильную стратегию: для числовых колонок использовали медиану (она устойчива к выбросам) или среднее, для категориальных и текстовых — константу «не указано», а пустые описания мы обработали отдельно. Вообще они должны были обрабатываться позже, но описания были уже готовы, поэтому пропуски мы тоже устранили пропуски: поставили название самого товара.
После обработки пропусков мы проверили данные на дубликаты — как полные, так и по ключевым полям вроде связки названия, бренда и цвета. К счастью, парсинг был настроен аккуратно, и дубликатов почти не оказалось.

В числовых колонках — особенно в ценах и количестве отзывов — обнаружились аномальные значения, которые могли исказить анализ. Мы применили winsorization: значения, выходящие за пределы полутора межквартильных размахов, не удалялись, а «прижимались» к границам допустимого диапазона. Это позволило сохранить все наблюдения, но снизить влияние экстремальных значений. Для количества отзывов дополнительно использовали логарифмирование — распределение было сильно скошено вправо, и логарифм сделал его ближе к нормальному. Отдельно проверили логические аномалии: отрицательные скидки, скидки больше ста процентов, рейтинги вне диапазона от одного до пяти. Такие ошибки были исправлены.

Затем мы закодировали категориальные признаки в числовые. Пол превратили в шкалу от 0 до 5, предусмотрев защиту от мусорных значений — названий моделей белья, которые по ошибке парсинга попали в эту колонку. Страны и бренды кодировали через frequency encoding: заменяли категорию на её долю в датасете. Это позволяет сохранить информацию о популярности категории без создания сотен лишних колонок. Для цвета выделили основной цвет и создали флаг мультиколора — признак того, что товар представлен в нескольких расцветках.

В завершение мы создали несколько расчётных признаков. Индекс ценности товара объединил рейтинг и логарифм количества отзывов — чтобы одним числом выразить востребованность. Ценовые сегменты разбили товары на четыре группы от эконома до премиума. Флаг бестселлера отметил топ-10 процентов товаров по количеству отзывов. Все эти метрики пригодились на следующих этапах анализа. Итогом очистки стал файл из 3708 строк и четырёх десятков колонок — полностью готовый к работе.

In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Загрузка данных
print("1. Загрузка данных")
file_path = 'wb_products_full_imputed.csv'

# Загрузика с указанием разделителя и кодировки
try:
    df = pd.read_csv(file_path, sep=',', encoding='utf-8')
    print(f"Файл успешно загружен. Размер данных: {df.shape}")
except UnicodeDecodeError:
    print("Попытка загрузки с кодировкой 'cp1251'")
    df = pd.read_csv(file_path, sep=',', encoding='cp1251')
    print(f"Файл успешно загружен. Размер данных: {df.shape}")

# 2. Приведение типов
print("\n2. Приведение данных к корректным типам")

# 2.1. Цены (удаляем пробелы, преобразуем в float)
if 'price' in df.columns:
    df['price'] = df['price'].astype(str).str.replace(' ', '').str.replace(',', '.').astype(float)

if 'price_no_discount' in df.columns:
    df['price_no_discount'] = df['price_no_discount'].astype(str).str.replace(' ', '').str.replace(',', '.').astype(float)

# 2.2. Процент скидки (удаляем знак %, преобразуем в float)
if 'discount_pct' in df.columns:
    df['discount_pct'] = df['discount_pct'].astype(str).str.replace('%', '').astype(float)

# 2.3. Рейтинг
if 'rating' in df.columns:
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

# 2.4. Количество отзывов
if 'reviews_count' in df.columns:
    df['reviews_count'] = pd.to_numeric(df['reviews_count'], errors='coerce').fillna(0).astype(int)

# 2.5. Рейтинг поставщика
if 'supplier_rating' in df.columns:
    df['supplier_rating'] = pd.to_numeric(df['supplier_rating'], errors='coerce')

# 2.6. Дата сбора
if 'collected_at' in df.columns:
    df['collected_at'] = pd.to_datetime(df['collected_at'], errors='coerce')

# 2.7. Номер страницы
if 'page' in df.columns:
    df['page'] = pd.to_numeric(df['page'], errors='coerce').fillna(0).astype(int)

# 2.8. Преобразование в категориальный тип (экономит память)
categorical_cols = ['brand', 'gender', 'country', 'color']

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

print("\nТипы данных после приведения:")
print(df.dtypes)

# 3. Первичный анализ пропусков
print("\n3. Анализ пропущенных значений")
# Количество пропусков в каждой колонке
missing_counts = df.isnull().sum()
# Процент пропусков
missing_percent = (missing_counts / len(df)) * 100

# Формирование DataFrame для удобства
missing_report = pd.DataFrame({
    'Колонка': df.columns,
    'Тип данных': df.dtypes.values,
    'Количество пропусков': missing_counts.values,
    'Процент пропусков': missing_percent.values
})

# Анализ механизма пропусков
print("\n3.1 Анализ механизма пропусков (MCAR / MAR / MNAR)")

# Проверка: связаны ли пропуски в одной колонке с другой
for col in ['care', 'country', 'brand', 'description']:
    if col in df.columns:
        missing_mask = df[col].isna()
        if missing_mask.sum() > 0:
            avg_price_missing = df[missing_mask]['price'].mean()
            avg_price_present = df[~missing_mask]['price'].mean()
            print(f"\n{col}:")
            print(f"  Пропусков: {missing_mask.sum()} ({missing_mask.sum()/len(df)*100:.1f}%)")
            print(f"  Цена при пропуске: {avg_price_missing:.0f}")
            print(f"  Цена при наличии: {avg_price_present:.0f}")
            
            # Если цена сильно отличается (>10%) - пропуски не случайны
            if abs(avg_price_missing - avg_price_present) > avg_price_present * 0.1:
                print(f"  Вывод: Пропуски НЕ случайны (MAR/MNAR)")
            else:
                print(f"  Вывод: Пропуски случайны (MCAR)")

# Сортировка по проценту пропусков (от большего к меньшему)
missing_report = missing_report.sort_values('Процент пропусков', ascending=False)
print("\nТоп-15 колонок с наибольшим количеством пропусков:")
print(missing_report.head(15).to_string(index=False))

# 3.2 Анализ колонок с пропусками и стратегия обработки
print("\n3.2 Анализ стратегии обработки для каждой колонки")

# Пока не трогаем эти колонки
columns_to_skip = ['reviews_text', 'ai_summary']

# Создание списков для хранения стратегий
columns_to_drop = []
columns_to_fill_mean = []
columns_to_fill_median = []
columns_to_fill_mode = []
columns_to_fill_const = []
columns_to_create_flag = []  # для MAR/MNAR колонок

# MAR/MNAR колонки (пропуски не случайны) — для них флаги
mnar_columns = ['care', 'country', 'brand']

# Колонки, которые заполняем константуой
columns_for_constant = [
    'country',      # страна производства
    'gender',       # пол (мода исказит редкие значения)
    'colors',       # цвета (множественные значения)
    'sizes',        # размеры (множественные значения)
    'description',  # текстовое описание
    'composition',  # состав ткани (текст)
    'care',         # уход (текст)
    'brand',        # бренд (много уникальных значений)
    'color',        # основной цвет
]

for col in missing_report['Колонка']:
    # Пропускаем колонки
    if col in columns_to_skip:
        print(f"  - Колонка '{col}' пропущена (не обрабатываю)")
        continue

    missing_pct = missing_report.loc[missing_report['Колонка'] == col, 'Процент пропусков'].values[0]
    dtype = missing_report.loc[missing_report['Колонка'] == col, 'Тип данных'].values[0]

    if missing_pct > 80:
        # Удаляем колонки с >80% пропусков
        print(f"  - Колонка '{col}' имеет {missing_pct:.1f}% пропусков. Удаляю.")
        columns_to_drop.append(col)
        
    elif missing_pct > 0 and missing_pct <= 80:
        # Для числовых колонок: заполняем средним/медианой
        if dtype in ['int64', 'float64']:
            if 'price' in col or 'rating' in col:
                print(f"  - Числовая колонка '{col}' ({missing_pct:.1f}% пропусков). Заполняем медианой.")
                columns_to_fill_median.append(col)
            else:
                print(f"  - Числовая колонка '{col}' ({missing_pct:.1f}% пропусков). Заполняем средним.")
                columns_to_fill_mean.append(col)
        else:
            # Категориальные/текстовые колонки
            if col in columns_for_constant:
                # Для MAR/MNAR колонок дополнительно создадим флаг
                if col in mnar_columns:
                    columns_to_create_flag.append(col)
                    print(f"  - Колонка '{col}' ({missing_pct:.1f}% пропусков). Заполняем константой + создаем флаг (MAR/MNAR).")
                else:
                    print(f"  - Колонка '{col}' ({missing_pct:.1f}% пропусков). Заполняем константой (MCAR).")
                columns_to_fill_const.append(col)
            else:
                # Только для действительно однородных колонок
                print(f"  - Колонка '{col}' ({missing_pct:.1f}% пропусков). Заполняем модой.")
                columns_to_fill_mode.append(col)

# 4. Применение выбранных стратегий
print("\n4. Применение выбранных стратегий")

# 4.1. Удаление колонок с большим количеством пропусков (>80%)
if columns_to_drop:
    original_columns = df.shape[1]
    df = df.drop(columns=columns_to_drop, errors='ignore')
    print(f"  - Удалено колонок: {len(columns_to_drop)}")
    print(f"    Удаленные колонки: {columns_to_drop}")
else:
    print(f"  - Колонок для удаления нет.")

# 4.2. Заполнение числовых колонок средним значением
if columns_to_fill_mean:
    for col in columns_to_fill_mean:
        if col in df.columns:
            mean_val = df[col].mean()
            df[col] = df[col].fillna(mean_val)
    print(f"  - Заполнено средним в колонках: {columns_to_fill_mean}")
else:
    print(f"  - Колонок для заполнения средним нет.")

# 4.3. Заполнение числовых колонок медианой (для цен и рейтингов)
if columns_to_fill_median:
    for col in columns_to_fill_median:
        if col in df.columns:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
    print(f"  - Заполнено медианой в колонках: {columns_to_fill_median}")
else:
    print(f"  - Колонок для заполнения медианой нет.")

# 4.4. Заполнение категориальных колонок
if columns_to_fill_const:
    for col in columns_to_fill_const:
        if col in df.columns:
            if col == 'description':
                df[col] = df[col].fillna(df['name'])
                print(f"  - {col}: заполнено названием товара")
            else:
                # Заполняем пропуски константой 'unknown'
                # Сначала преобразуем в object, чтобы избежать проблем с category
                if df[col].dtype.name == 'category':
                    df[col] = df[col].astype(object)
                df[col] = df[col].fillna('unknown')
                print(f"  - {col}: заполнено 'unknown'")
    print(f"  - Обработаны колонки: {columns_to_fill_const}")
else:
    print(f"  - Колонок для обработки нет.")

# 4.5. Заполнение категориальных колонок модой (для однородных колонок)
if columns_to_fill_mode:
    for col in columns_to_fill_mode:
        if col in df.columns:
            mode_val = df[col].mode()
            if not mode_val.empty:
                df[col] = df[col].fillna(mode_val[0])
            else:
                df[col] = df[col].fillna('missing')
    print(f"  - Заполнено модой в колонках: {columns_to_fill_mode}")
else:
    print(f"  - Колонок для заполнения модой нет.")

# 5. Финальная проверка
print("\n5. Финальная проверка")

# Проверяем все колонки (кроме тех, которые мы сознательно пропустили)
all_columns = [col for col in df.columns if col not in columns_to_skip]

final_missing = df[all_columns].isnull().sum().sum()
print(f"Общее количество пропусков в обработанных колонках: {final_missing}")

if final_missing == 0:
    print("Данные успешно очищены от пропусков!")
else:
    print("В данных все еще есть пропуски. Проверить следующие колонки:")
    remaining = df[all_columns].isnull().sum()
    print(remaining[remaining > 0])

# Проверяем пропуски в пропущенных колонках (для информации)
print(f"\nПропуски в пропущенных колонках (не обрабатывались):")
for col in columns_to_skip:
    if col in df.columns:
        null_cnt = df[col].isnull().sum()
        print(f"  - {col}: {null_cnt} пропусков ({null_cnt/len(df)*100:.1f}%)")

# 6. Проверка на дубликаты
print("\n6. Проверка на дубликаты")

# 6.1. Проверка полных дубликатов (все колонки одинаковые)
full_duplicates = df.duplicated().sum()
print(f"Полных дубликатов строк (все колонки совпадают): {full_duplicates}")

# 6.2. Проверка дубликатов по ключевым полям (product_id - уникальный идентификатор)
if 'product_id' in df.columns:
    product_id_duplicates = df['product_id'].duplicated().sum()
    print(f"Дубликатов по product_id: {product_id_duplicates}")
    
    # Примеры дубликатов по product_id
    if product_id_duplicates > 0:
        duplicated_ids = df[df['product_id'].duplicated()]['product_id'].tolist()[:5]
        print(f"  Примеры дублирующихся product_id: {duplicated_ids}")
        
        # Строки с первым дублирующимся ID
        first_dup_id = duplicated_ids[0] if duplicated_ids else None
        if first_dup_id:
            print(f"\n  Пример дубликата (product_id = {first_dup_id}):")
            print(df[df['product_id'] == first_dup_id].to_string())
else:
    print("Колонка 'product_id' не найдена")

# 6.3. Проверка дубликатов по связке (name + brand + color) - может указывать на одинаковые товары
if 'name' in df.columns and 'brand' in df.columns:
    # Создаем временную колонку для проверки
    duplicate_key_cols = ['name', 'brand']
    if 'color' in df.columns:
        duplicate_key_cols.append('color')
    
    semantic_duplicates = df.duplicated(subset=duplicate_key_cols, keep=False).sum()
    print(f"\nПотенциальных дубликатов по ({', '.join(duplicate_key_cols)}): {semantic_duplicates}")
    
    if semantic_duplicates > 0:
        # Показываем пример группы дубликатов
        dup_mask = df.duplicated(subset=duplicate_key_cols, keep=False)
        sample_dups = df[dup_mask].head(4)
        print(f"\nПримеры потенциальных дубликатов (первые 4 строки):")
        print(sample_dups[duplicate_key_cols + ['price']].to_string())

# 6.4. Статистика по дубликатам в разрезе
print("Статистика по дубликатам:")
print(f"  - Всего строк в данных: {len(df)}")
print(f"  - Полных дубликатов: {full_duplicates}")
if 'product_id' in df.columns:
    print(f"  - Уникальных product_id: {df['product_id'].nunique()}")
    print(f"  - Потеряно уникальных ID из-за дубликатов: {len(df) - df['product_id'].nunique()}")

# 7. Анализ выбросов
print("\n7. Анализ выбросов в числовых колонках")

numeric_cols = ['price', 'price_no_discount', 'discount_pct', 'rating', 'reviews_count', 'supplier_rating']

# 7.1. Базовая статистика
print("\n7.1. Базовая статистика:")
print(df[numeric_cols].describe().round(2))

# 7.2. Поиск выбросов методом IQR
print("\n7.2. Поиск выбросов методом IQR:")

outliers_summary = []

for col in numeric_cols:
    if col not in df.columns:
        continue
    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outliers_count = len(outliers)
    
    outliers_summary.append({
        'Колонка': col,
        'Выбросов': outliers_count,
        'Процент': round(outliers_count / len(df) * 100, 2),
        'Нижняя граница': round(lower_bound, 2),
        'Верхняя граница': round(upper_bound, 2)
    })
    
    print(f"{col}: {outliers_count} выбросов ({outliers_count/len(df)*100:.1f}%)")

# Сводная таблица IQR
outliers_df = pd.DataFrame(outliers_summary)
print("\nСводная таблица выбросов (IQR):")
print(outliers_df.to_string(index=False))

# 7.3. Поиск выбросов методом Z-score (для сравнения)
print("\n7.3. Поиск выбросов методом Z-score (|Z| > 3):")

from scipy import stats

for col in ['price', 'discount_pct']:
    if col not in df.columns:
        continue
    
    values = df[col].dropna().values
    
    if len(values) > 1:
        z_scores = np.abs(stats.zscore(values))
        outliers_count = (z_scores > 3).sum()
        
        print(f"{col}: {outliers_count} выбросов ({outliers_count/len(values)*100:.2f}%)")
        print(f"   Среднее: {values.mean():.2f}, Стандартное отклонение: {values.std():.2f}")

# 7.4. Логические проверки (аномалии)
print("\n7.4. Логические проверки:")

if 'discount_pct' in df.columns:
    print(f"   Отрицательные скидки: {(df['discount_pct'] < 0).sum()}")
    print(f"   Скидки >100%: {(df['discount_pct'] > 100).sum()}")

if 'rating' in df.columns:
    invalid_rating = (df['rating'] < 1) | (df['rating'] > 5)
    print(f"   Некорректных рейтингов (<1 или >5): {invalid_rating.sum()}")

if 'price' in df.columns and 'price_no_discount' in df.columns:
    print(f"   Цена со скидкой > цены без скидки: {(df['price'] > df['price_no_discount']).sum()}")

# 7.5. Обработка выбросов (Winsorization для price)
if 'price' in df.columns:
    Q1 = df['price'].quantile(0.25)
    Q3 = df['price'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers_count = ((df['price'] < lower) | (df['price'] > upper)).sum()
    print(f"\n7.5. Обработка price: {outliers_count} выбросов ({outliers_count/len(df)*100:.1f}%)")
    
    df['price_winsorized'] = df['price'].clip(lower=lower, upper=upper)
    print(f"   Среднее: {df['price'].mean():.0f} -> {df['price_winsorized'].mean():.0f}")
    print(f"   Создана колонка price_winsorized")

# Для price_no_discount тоже можно сделать Winsorization
if 'price_no_discount' in df.columns:
    Q1 = df['price_no_discount'].quantile(0.25)
    Q3 = df['price_no_discount'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    df['price_no_discount_winsorized'] = df['price_no_discount'].clip(lower=lower, upper=upper)
    print(f"   Создана колонка price_no_discount_winsorized")
    
# 7.6. Анализ рейтингов
if 'rating' in df.columns and 'reviews_count' in df.columns:
    zero_rating = (df['rating'] == 0.0).sum()
    weird = ((df['rating'] == 0.0) & (df['reviews_count'] > 0)).sum()
    
    print(f"\n7.6. Анализ рейтингов:")
    print(f"   rating = 0.0: {zero_rating} товаров")
    
    if weird > 0:
        print(f"   ОШИБКА: {weird} товаров с rating=0.0, но есть отзывы!")
    else:
        print(f"   rating=0.0 означает отсутствие оценок (корректно)")

# 7.7. Логарифмирование для reviews_count
print("\n7.7. Логарифмирование для нормализации:")

if 'reviews_count' in df.columns:
    df['reviews_count_log'] = np.log1p(df['reviews_count'])
    print(f"   reviews_count_log: логарифмическая версия создана")
    print(f"   Диапазон: {df['reviews_count_log'].min():.2f} - {df['reviews_count_log'].max():.2f}")

# 7.8. Обработка выбросов для всех числовых колонок
print("\n7.8. Обработка выбросов для числовых колонок:")

# 1. Для price (уже есть winsorized)
if 'price_winsorized' in df.columns:
    print(f"   price: уже обработана (winsorized)")

# 2. Для discount_pct - очистка и winsorization
if 'discount_pct' in df.columns:
    # Сначала очищаем от некорректных значений (0-100%)
    invalid_discounts = ((df['discount_pct'] < 0) | (df['discount_pct'] > 100)).sum()
    if invalid_discounts > 0:
        df['discount_pct'] = df['discount_pct'].clip(lower=0, upper=100)
        print(f"   discount_pct: исправлено {invalid_discounts} некорректных скидок")
    
    # Затем winsorization для выбросов
    Q1 = df['discount_pct'].quantile(0.25)
    Q3 = df['discount_pct'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers_count = ((df['discount_pct'] < lower) | (df['discount_pct'] > upper)).sum()
    if outliers_count > 0:
        df['discount_pct_winsorized'] = df['discount_pct'].clip(lower=lower, upper=upper)
        print(f"   discount_pct_winsorized: {outliers_count} выбросов ограничено")
    else:
        df['discount_pct_winsorized'] = df['discount_pct']
        print(f"   discount_pct: выбросов нет")

# 3. Для rating - заменяем 0.0 на NaN (товары без отзывов) и winsorization
if 'rating' in df.columns:
    # Заменяем 0.0 на NaN (это не выбросы, а маркер "нет оценок")
    zero_count = (df['rating'] == 0.0).sum()
    if zero_count > 0:
        df['rating_clean'] = df['rating'].replace(0.0, np.nan)
        print(f"   rating: {zero_count} нулевых рейтингов заменены на NaN")
    else:
        df['rating_clean'] = df['rating']
    
    # Winsorization для rating (исключая NaN)
    rating_clean_no_nan = df['rating_clean'].dropna()
    if len(rating_clean_no_nan) > 0:
        Q1 = rating_clean_no_nan.quantile(0.25)
        Q3 = rating_clean_no_nan.quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        outliers_count = ((rating_clean_no_nan < lower) | (rating_clean_no_nan > upper)).sum()
        if outliers_count > 0:
            df['rating_winsorized'] = df['rating_clean'].clip(lower=lower, upper=upper)
            print(f"   rating_winsorized: {outliers_count} выбросов ограничено")
        else:
            df['rating_winsorized'] = df['rating_clean']
            print(f"   rating: выбросов нет")

# 4. Для supplier_rating - winsorization
if 'supplier_rating' in df.columns:
    Q1 = df['supplier_rating'].quantile(0.25)
    Q3 = df['supplier_rating'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers_count = ((df['supplier_rating'] < lower) | (df['supplier_rating'] > upper)).sum()
    if outliers_count > 0:
        df['supplier_rating_winsorized'] = df['supplier_rating'].clip(lower=lower, upper=upper)
        print(f"   supplier_rating_winsorized: {outliers_count} выбросов ограничено")
    else:
        df['supplier_rating_winsorized'] = df['supplier_rating']
        print(f"   supplier_rating: выбросов нет")

# 5. Для reviews_count - уже есть логарифмическая версия
if 'reviews_count_log' in df.columns:
    print(f"   reviews_count: уже обработана (логарифмирование)")

# 7.9. Итоговая статистика по обработанным колонкам
print("\n7.9. Итоговая статистика после обработки:")

processed_cols = []
if 'price_winsorized' in df.columns:
    processed_cols.append('price_winsorized')
if 'price_no_discount_winsorized' in df.columns:
    processed_cols.append('price_no_discount_winsorized')
if 'reviews_count_log' in df.columns:
    processed_cols.append('reviews_count_log')
if 'discount_pct_winsorized' in df.columns:
    processed_cols.append('discount_pct_winsorized')
if 'rating_winsorized' in df.columns:
    processed_cols.append('rating_winsorized')
if 'supplier_rating_winsorized' in df.columns:
    processed_cols.append('supplier_rating_winsorized')

if processed_cols:
    print(df[processed_cols].describe().round(2))
    
print("\nОбработка выбросов завершена!")

    
# 8. Кодирование категориальных признаков
print("8. Кодирование категориальных признаков")

from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# 8.1 Кодирование GENDER
print("\n1. Кодирование GENDER")

gender_mapping = {
    'Женский': 0,
    'Мужской': 1,
    'Девочки': 2,
    'Мальчики': 3,
    'Детский': 4,
    'Не указан': 5
}

def encode_gender(value):
    # Проверка на NaN
    if pd.isna(value):
        return 5  # NaN → 5 (Не указан)
    
    value_str = str(value).strip()
    
    # Прямое соответствие
    if value_str in gender_mapping:
        return gender_mapping[value_str]
    
    # Проверка на пустую строку или "не указан" в разных вариациях
    value_lower = value_str.lower()
    if value_str == '' or value_lower in ['не указан', 'не указано', 'неизвестно', 'none', 'null']:
        return 5
    
    # Нечеткое соответствие
    if 'жен' in value_lower:
        return 0
    elif 'муж' in value_lower:
        return 1
    elif 'девоч' in value_lower:
        return 2
    elif 'мальч' in value_lower:
        return 3
    elif 'дет' in value_lower:
        return 4
    
    # Всё остальное тоже считаем "Не указан"
    return 5

# Применяем кодирование
df['gender_encoded'] = df['gender'].apply(encode_gender)

# ПРИНУДИТЕЛЬНО заменяем все оставшиеся NaN на 5
df['gender_encoded'] = df['gender_encoded'].fillna(5)

print(f"\n   Результат кодирования gender:")
for code in sorted(df['gender_encoded'].unique()):
    count = (df['gender_encoded'] == code).sum()
    if code == 0:
        name = 'Женский'
    elif code == 1:
        name = 'Мужской'
    elif code == 2:
        name = 'Девочки'
    elif code == 3:
        name = 'Мальчики'
    elif code == 4:
        name = 'Детский'
    elif code == 5:
        name = 'Не указан'
    else:
        name = 'other'
    print(f"      {code} ({name}): {count} ({count/len(df)*100:.1f}%)")

# 8.2 Кодирование COUNTRY
print("\n2. Кодирование COUNTRY")

# Получаем все уникальные страны
unique_countries = sorted([str(c) for c in df['country'].dropna().unique()])
country_mapping = {country: idx for idx, country in enumerate(unique_countries)}

def encode_country(value):
    if pd.isna(value):
        return 999
    value_str = str(value).strip()
    if value_str == '':
        return 999
    return country_mapping.get(value_str, 999)

df['country_encoded'] = df['country'].apply(encode_country)

print(f"\n   Результат кодирования country:")
print(f"   Уникальных стран: {len(unique_countries)}")
print(f"   Не указано: {(df['country_encoded'] == 999).sum()}")

print(f"\n   Топ-10 стран:")
for country, count in df['country'].value_counts().head(10).items():
    code = country_mapping.get(str(country), 999)
    print(f"      {code} ({country}): {count}")
    
# 8.3 Обработка COLOR
print("\n3. Обработка COLOR")

# Основной цвет (первый из списка)
df['color_main'] = df['color'].apply(
    lambda x: str(x).split(';')[0].strip() if pd.notna(x) else 'unknown'
)

# Количество цветов в товаре
df['color_count'] = df['color'].apply(
    lambda x: len(str(x).split(';')) if pd.notna(x) else 0
)

# Флаг мультиколор
df['is_multicolor'] = (df['color_count'] > 1).astype(int)

# Частотное кодирование основного цвета
freq = df['color_main'].value_counts(normalize=True).to_dict()
df['color_freq'] = df['color_main'].map(freq).fillna(0)

print(f"   Уникальных цветов было: {df['color'].nunique()}")
print(f"   Уникальных основных цветов стало: {df['color_main'].nunique()}")
print(f"   Примеры основных цветов: {dict(df['color_main'].value_counts().head(5))}")

# 8.4 Кодирование остальных категорий
print("\n4. Кодирование остальных категорий")

# Brand (Frequency Encoding)
if 'brand' in df.columns:
    freq = df['brand'].value_counts(normalize=True).to_dict()
    df['brand_freq'] = df['brand'].map(freq).fillna(0)
    print(f"   brand -> brand_freq (Frequency Encoding, {df['brand'].nunique():,} брендов → 1 признак)")

# Subject_id (Frequency Encoding)
if 'subject_id' in df.columns:
    freq = df['subject_id'].value_counts(normalize=True).to_dict()
    df['subject_id_freq'] = df['subject_id'].map(freq).fillna(0)
    print(f"   subject_id -> subject_id_freq (Frequency Encoding, {df['subject_id'].nunique()} категорий → 1 признак)")

# Category_query (Frequency Encoding)
if 'category_query' in df.columns:
    freq = df['category_query'].value_counts(normalize=True).to_dict()
    df['category_query_freq'] = df['category_query'].map(freq).fillna(0)
    print(f"   category_query -> category_query_freq (Frequency Encoding, {df['category_query'].nunique()} категорий → 1 признак)")

# 8.5 Итоги
print("Итоги кодирования")

encoded_cols = ['gender_encoded', 'country_encoded', 'color_freq', 'color_count', 'is_multicolor', 
                'brand_freq', 'subject_id_freq', 'category_query_freq']
encoded_cols = [c for c in encoded_cols if c in df.columns]

print(f"\nСоздано закодированных признаков: {len(encoded_cols)}")
print(f"   Список: {encoded_cols}")

print("\nПример закодированных значений (первые 5 строк):")
sample_cols = ['gender', 'gender_encoded', 'country', 'country_encoded', 'color_main', 'color_freq']
sample_cols = [c for c in sample_cols if c in df.columns]
if sample_cols:
    print(df[sample_cols].head(5).to_string())

# 9. Дополнительно: Новые расчетные признаки
print("\n9. Новые расчетные признаки")

# 1. Индекс ценности товара
if 'rating_clean' in df.columns and 'reviews_count_log' in df.columns:
    df['product_value'] = df['rating_clean'] * df['reviews_count_log']
    df['product_value'] = df['product_value'].fillna(0)  # 0 для товаров без отзывов
    print("   product_value = рейтинг * log(отзывы)")

# 2. Эффективность скидки
if 'discount_pct_winsorized' in df.columns and 'rating_clean' in df.columns:
    df['discount_score'] = df['discount_pct_winsorized'] / (df['rating_clean'] + 0.1)
    df['discount_score'] = df['discount_score'].fillna(0)
    print("   discount_score = скидка / рейтинг")

# 3. Категория цены
if 'price_winsorized' in df.columns:
    df['price_tier'] = pd.qcut(df['price_winsorized'], q=4, labels=['Эконом', 'Стандарт', 'Бизнес', 'Премиум'])
    print("   price_tier = квартиль цены")

# 4. Соотношение цены со скидкой и без
if 'price_winsorized' in df.columns and 'price_no_discount_winsorized' in df.columns:
    df['savings_percent'] = (df['price_no_discount_winsorized'] - df['price_winsorized']) / df['price_no_discount_winsorized'] * 100
    print("   savings_percent = реальный процент экономии")

# 5. Флаг "хит продаж" (топ-10 по отзывам)
if 'reviews_count' in df.columns:
    top_10_percent = df['reviews_count'].quantile(0.9)
    df['is_bestseller'] = (df['reviews_count'] > top_10_percent).astype(int)
    print("   is_bestseller = топ-10 по количеству отзывов")

# 10. Сохранение обработанного датасета
print("10. Сохранение обработанных данных")

# Сохраняем в Excel
output_file = 'wb_products_processed.xlsx'

try:
    df.to_excel(output_file, index=False, engine='openpyxl')
    print(f"Данные сохранены в файл: {output_file}")
    print(f"   Размер: {df.shape[0]} строк × {df.shape[1]} столбцов")
except Exception as e:
    print(f"Ошибка при сохранении в Excel: {e}")
    print("   Попытка сохранить в CSV")
    
    # CSV
    output_csv = 'wb_products_processed.csv'
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"Данные сохранены в файл: {output_csv}")

# Также можно сохранить в CSV (более универсальный формат)
#print("\nДополнительная выгрузка:")
#df.to_csv('wb_products_processed.csv', index=False, encoding='utf-8-sig')
#print("   CSV файл: wb_products_processed.csv")

1. Загрузка данных
Файл успешно загружен. Размер данных: (3708, 24)

2. Приведение данных к корректным типам

Типы данных после приведения:
product_id                    int64
imt_id                        int64
name                         object
brand                      category
price                       float64
price_no_discount           float64
discount_pct                float64
rating                      float64
reviews_count                 int64
subject_id                    int64
supplier_rating             float64
sizes                        object
colors                       object
category_query               object
page                          int64
collected_at         datetime64[ns]
description                  object
composition                  object
color                      category
gender                     category
country                    category
care                         object
ai_summary                   object
reviews_text                 obj

Далее занялись обработкой текстовых данных: она была чуть более простой, так как было принято решение не заполнять пропуски. Во-первых, это нормально, что у каких-то товаров нет отзывов, во-вторых, будет очень некорректно заполнять столь объемные текстовые данные за других людей. Это не даст нам никаких объективных результатов. Поэтому мы приняли решение, что 2500 данных нам достаточно для анализа.

Первым делом мы очистили тексты от технического мусора: убрали табуляции, переносы строк, множественные пробелы. Для каждого поля создали две версии — очищенную (для подсчёта длины) и в нижнем регистре (для поиска слов). Затем посчитали базовые метрики: длину в символах и словах, количество предложений, среднюю длину предложения. Для описаний дополнительно рассчитали индекс читаемости Флеша-Кинкейда, адаптированный для русского языка — он показывает, насколько легко или сложно читается текст. Отзывы, которые хранились склеенными через разделитель, разбили на отдельные тексты и посчитали их количество и среднюю длину.

Дальше решили сделать словари. Мы вручную составили списки корней слов для разных категорий: сенсорная лексика (шёлк, кружево, мягкий, облегающий), эмоциональная (шикарный, неотразимый, божественный), статусная (премиум, люкс, итальянский), манипулятивная (хит, лимитированный, успей, акция). Отдельные словари — для усилителей, сезонных слов, телесной лексики, конкретных и абстрактных терминов. Корни подобраны так, чтобы ловить разные окончания: например, корень «шёлк» найдёт и «шёлковое», и «шёлка», и «шёлку».

Для каждого названия и описания мы посчитали плотность каждой категории — долю слов из словаря среди всех слов текста. Для отзывов добавили метрику субъективности — долю личных местоимений «я», «мне», «мой», «моя» — и плотность телесной лексики. Создали бинарные флаги: женский или мужской товар, глубокая ли скидка, есть ли отзывы. В итоге получилось больше двадцати новых признаков, которые затем использовались для проверки когнитивных гипотез.

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Загрузка обработанного на прошлом шаге датасета
df = pd.read_excel("wb_products_processed.xlsx", engine='openpyxl')
print(f"Загружено: {len(df)} товаров, {len(df.columns)} колонок")

# Функции очистки текста: убираем табуляции, переносы, лишние пробелы
def clean_text(text):
    if pd.isna(text) or text == '':
        return ''
    text = str(text)
    text = text.replace('\t', ' ').replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def clean_text_lower(text):
    return clean_text(text).lower()

# Очистка всех текстовых колонок и создание дополнительных версий
for col in ['name', 'description', 'reviews_text', 'ai_summary']:
    if col not in df.columns:
        continue
    df[col] = df[col].fillna('')                       # NaN - пустая строка
    df[col + '_clean'] = df[col].apply(clean_text)     # очищенная версия
    df[col + '_lower'] = df[col].apply(clean_text_lower)  # для поиска слов
    df[col + '_len_chars'] = df[col + '_clean'].apply(len)
    df[col + '_len_words'] = df[col + '_clean'].apply(lambda x: len(x.split()) if x else 0)

# Метрики описаний: предложения, средняя длина предложения, читаемость
df['desc_sentences'] = df['description_clean'].apply(
    lambda x: len(re.split(r'[.!?]+', x)) if x else 1
)
df['desc_avg_sentence_len'] = np.where(
    df['desc_sentences'] > 0,
    df['description_len_words'] / df['desc_sentences'], 0
)
# Индекс читаемости Флеша-Кинкейда (адаптирован для русского)
df['desc_readability'] = np.where(
    (df['desc_sentences'] > 0) & (df['description_len_words'] > 0),
    206.835 - 1.3 * (df['description_len_words'] / df['desc_sentences'])
            - 60.1 * ((df['description_len_chars'] / df['description_len_words']) / 3),
    np.nan
)

# Разбивка отзывов (хранятся через |||) на списки и базовая статистика
df['reviews_list'] = df['reviews_text_clean'].apply(
    lambda x: [r.strip() for r in x.split('|||') if r.strip()] if x else []
)
df['review_count'] = df['reviews_list'].apply(len)
df['review_main'] = df['reviews_list'].apply(lambda x: x[0] if x else '')
df['review_main_len'] = df['review_main'].apply(len)
df['review_avg_len'] = df['reviews_list'].apply(
    lambda x: np.mean([len(r) for r in x]) if x else 0
)

# Словари для поиска когнитивных паттернов (корни слов, ловят разные окончания)
SENSORY = [
    'шёлк', 'кружев', 'мягк', 'облега', 'воздушн', 'нежн', 'атласн',
    'гладк', 'струящ', 'дышащ', 'приятн', 'тактильн', 'фактур', 'скольз',
    'пушист', 'бархат', 'велюр', 'кашемир', 'трикотаж', 'хлопк', 'лён',
    'шелковист', 'невесом', 'лайкра', 'стрейч', 'эластич', 'тянущ',
]

EMOTIONAL = [
    'шикарн', 'праздничн', 'неотразим', 'любовь', 'мечта', 'роскошн',
    'великолепн', 'потрясающ', 'восхитительн', 'божествен', 'идеальн',
    'эффектн', 'ослепительн', 'сногсшибательн', 'изумительн', 'волшебн',
    'сказочн', 'фееричн', 'прекрасн', 'чудесн', 'красивейш', 'обалденн',
    'бомбическ', 'потрясн', 'крутейш', 'стильн', 'модн', 'трендов',
]

STATUS = [
    'премиум', 'люкс', 'итальян', 'француз', 'дизайнер', 'элитн',
    'эксклюзив', 'бренд', 'статус', 'дорог', 'престиж', 'vip',
    'лимитирован', 'коллекцион', 'ручн', 'авторск',
]

MANIPULATIVE = [
    'хит', 'лимитир', 'новинк', 'выбор', 'тренд', 'последн',
    'специальн', 'только', 'успей', 'акци', 'распродаж', 'скидк',
    'суперцен', 'выгод', 'шанс', 'недели', 'месяца', 'сезон',
    'бестселлер', 'musthave', 'must have', 'hot', 'top',
]

SEASONAL = [
    'летн', 'зимн', 'весенн', 'осенн', 'демисезон', 'тёпл',
    'утеплён', 'облегчён', 'жара', 'мороз', 'холод',
]

INTENSIFIERS = [
    'очень', 'макси', 'ультра', 'супер', 'мега', 'гипер', 'экстра',
    'абсолютн', 'совершенн', 'невероятн', 'предельн', 'максимальн',
    'тотал', 'тотально', 'безумн', 'дико', 'жутко', 'страшно',
]

BODY_WORDS = [
    'сидит', 'облегает', 'стройнит', 'полнит', 'бёдра', 'талия',
    'плечи', 'грудь', 'фигура', 'живот', 'ноги', 'руки', 'спина',
    'посадка', 'размер', 'рост', 'длина', 'ширина', 'объём',
    'подчеркива', 'скрыва', 'украша', 'вытягива',
]

CONCRETE_WORDS = [
    'ткань', 'швы', 'размер', 'посадка', 'пуговиц', 'молния',
    'рукав', 'ворот', 'карман', 'подол', 'пояс', 'манжет',
    'строчк', 'прошит', 'фурнитур', 'подклад', 'хлопок', 'шерсть',
]

ABSTRACT_WORDS = [
    'элегантность', 'стиль', 'образ', 'комфорт', 'изысканность',
    'индивидуальность', 'женственность', 'изящество', 'грация',
    'шарм', 'харизма', 'атмосфера', 'настроение', 'впечатление',
    'уверенность', 'чувство', 'ощущение', 'лёгкость',
]

# Функции подсчёта: количество совпадений и плотность (доля от всех слов)
def count_matches(text_lower, word_roots):
    if not text_lower:
        return 0
    count = 0
    for word in text_lower.split():
        for root in word_roots:
            if root in word:
                count += 1
                break
    return count

def density(text_lower, word_roots):
    if not text_lower:
        return 0.0
    words = text_lower.split()
    if len(words) == 0:
        return 0.0
    return count_matches(text_lower, word_roots) / len(words)

# Признаки из названий: плотности разных типов лексики и доля английских слов
print("\nИзвлечение признаков из названий...")

df['name_density_sensory'] = df['name_lower'].apply(lambda x: density(x, SENSORY))
df['name_density_emotional'] = df['name_lower'].apply(lambda x: density(x, EMOTIONAL))
df['name_density_status'] = df['name_lower'].apply(lambda x: density(x, STATUS))
df['name_density_manipulative'] = df['name_lower'].apply(lambda x: density(x, MANIPULATIVE))
df['name_density_seasonal'] = df['name_lower'].apply(lambda x: density(x, SEASONAL))
df['name_density_intensifiers'] = df['name_lower'].apply(lambda x: density(x, INTENSIFIERS))
df['name_english_ratio'] = df['name_lower'].apply(
    lambda x: sum(1 for c in x if c.isascii() and c.isalpha()) / 
              max(sum(1 for c in x if c.isalpha()), 1) if x else 0
)

print(f"  Сенсорная: {df['name_density_sensory'].mean():.3f}")
print(f"  Эмоциональная: {df['name_density_emotional'].mean():.3f}")
print(f"  Статусная: {df['name_density_status'].mean():.3f}")
print(f"  Манипулятивная: {df['name_density_manipulative'].mean():.3f}")

# Признаки из описаний: сенсорная плотность, конкретность vs абстрактность
print("\nИзвлечение признаков из описаний...")

df['desc_density_sensory'] = df['description_lower'].apply(lambda x: density(x, SENSORY))
df['desc_density_emotional'] = df['description_lower'].apply(lambda x: density(x, EMOTIONAL))
df['desc_density_manipulative'] = df['description_lower'].apply(lambda x: density(x, MANIPULATIVE))
df['desc_concrete_count'] = df['description_lower'].apply(lambda x: count_matches(x, CONCRETE_WORDS))
df['desc_abstract_count'] = df['description_lower'].apply(lambda x: count_matches(x, ABSTRACT_WORDS))
df['desc_concreteness_ratio'] = np.where(
    (df['desc_concrete_count'] + df['desc_abstract_count']) > 0,
    df['desc_concrete_count'] / (df['desc_concrete_count'] + df['desc_abstract_count']), 0.5
)

print(f"  Сенсорная: {df['desc_density_sensory'].mean():.3f}")
print(f"  Конкретность: {df['desc_concreteness_ratio'].mean():.3f}")

# Признаки из отзывов: субъективность и телесная лексика
print("\nИзвлечение признаков из отзывов...")

df['review_main'] = df['review_main'].fillna('')
df['review_subjectivity'] = df['review_main'].apply(
    lambda x: (str(x).lower().count('я ') + str(x).lower().count('мне ') +
               str(x).lower().count('мой ') + str(x).lower().count('моя ')) /
              max(len(str(x).split()), 1) if str(x).strip() else 0
)
df['review_body_density'] = df['review_main'].apply(
    lambda x: density(str(x).lower() if str(x).strip() else '', BODY_WORDS)
)
df['review_avg_len'] = df['review_avg_len'].fillna(0)

print(f"  Субъективность: {df['review_subjectivity'].mean():.3f}")
print(f"  Телесная лексика: {df['review_body_density'].mean():.3f}")

# Бинарные признаки для проверки гипотез
if 'gender_encoded' in df.columns:
    df['is_female'] = (df['gender_encoded'] == 0).astype(int)
    df['is_male'] = (df['gender_encoded'] == 1).astype(int)
if 'discount_pct' in df.columns:
    df['is_deep_discount'] = (df['discount_pct'] > 50).astype(int)
df['has_reviews'] = (df['review_count'] > 0).astype(int)

# Сохранение результата
output_file = "wb_nlp_full.xlsx"
df.to_excel(output_file, index=False, engine='openpyxl')

nlp_cols = [c for c in df.columns if c.startswith(('name_density', 'name_english', 'desc_density', 
    'desc_concreteness', 'desc_readability', 'review_subjectivity', 'review_body', 
    'review_avg_len', 'review_count', 'is_female', 'is_male', 'is_deep_discount', 'has_reviews'))]

print(f"\nСохранено: {output_file}")
print(f"NLP-признаков: {len(nlp_cols)}")
print(f"Всего колонок: {len(df.columns)}")

Загружено: 3708 товаров, 45 колонок

Извлечение признаков из названий...
  Сенсорная: 0.018
  Эмоциональная: 0.005
  Статусная: 0.001
  Манипулятивная: 0.010

Извлечение признаков из описаний...
  Сенсорная: 0.016
  Конкретность: 0.405

Извлечение признаков из отзывов...
  Субъективность: 0.032
  Телесная лексика: 0.035

Сохранено: wb_nlp_full.xlsx
NLP-признаков: 20
Всего колонок: 88
